# Transcript (Text) Preprocessing — OpenSLR-52

Applies the same text-cleaning pipeline as `transcript.ipynb` (NFC normalization,
whitespace/punctuation cleanup, emoji removal, character whitelist, pure-English
flagging) to the OpenSLR-52 Sinhala corpus, instead of the Linga/YouTube/BizBrains
combined dataset.

**Two things are different here vs. `transcript.ipynb`, both because of how
`audioPreprocess/audio_openslr.ipynb` produced this source:**

1. **Source is a loose file tree, not one embedded-bytes parquet.** OpenSLR-52 is
   read from `data/raw/openslr_52/processed/openslr_52_clean/` -- deduplicated +
   VAD-trimmed `.flac` files plus `utt_spk_text.tsv` (`file_id`, `speaker_id`,
   `transcript`) -- rather than a `text` column already sitting in a parquet-backed
   dataframe. This notebook reads that TSV into the same `text` / `source_dataset`
   shape the rest of the pipeline expects (adding `source_dataset = "openslr"` for
   parity with the other three sources), then runs the identical cleaning steps.
2. **Pure-English rows were already removed upstream.** `audio_openslr.ipynb`
   dropped pure-English clips (both the `.flac` file and the TSV row) as its own
   final step. Step 6 below re-runs the same check purely as a safety net -- it
   should find ~0 rows, not because the logic differs, but because the corpus
   should already be clean of them.

**Not included here:** the embedded-WAV-bytes `final_dataset.parquet` export used
for the other three sources. OpenSLR-52 is ~150k clips (~10x the combined
Linga/YouTube/BizBrains corpus), and `audio_openslr.ipynb`'s own notes explain it
deliberately moved *away* from an embedded-bytes parquet approach for this dataset
to avoid the OOM issues that scale caused. This notebook instead writes cleaned
text back out as a companion TSV (`utt_spk_text_cleaned.tsv`) next to the existing
`.flac` tree, leaving the audio files themselves untouched. Say the word if you
actually want a bytes-embedded parquet built for this dataset too, despite the size.

In [1]:
import os
import re
import unicodedata
from collections import Counter

import pandas as pd

# Reads the already deduped + VAD-trimmed OpenSLR-52 tree produced by
# audio_openslr.ipynb -- loose .flac files + this TSV, not a single parquet.
CLEAN_DIR = "../../../../data/raw/openslr_52/processed/openslr_52_clean"
TSV_PATH = os.path.join(CLEAN_DIR, "utt_spk_text.tsv")

tsv_df = pd.read_csv(TSV_PATH, sep="\t", header=None, names=["file_id", "speaker_id", "text"])
tsv_df["source_dataset"] = "openslr"

# Named df_trim (not df_raw) to match the rest of this pipeline's variable names --
# the VAD-based silence trim already happened upstream in audio_openslr.ipynb.
df_trim = tsv_df
print(f"Loaded {TSV_PATH}  ->  {df_trim.shape}")
df_trim[["source_dataset", "text"]].head(5)

Loaded ../../../../data/raw/openslr_52/processed/openslr_52_clean/utt_spk_text.tsv  ->  (150191, 4)


,source_dataset,text
0,openslr,මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී
1,openslr,උන්වහන්සේ කපාපු
2,openslr,එය එතනින් අවසන් නොවී
3,openslr,සිතින් අයහපතෙහි හැසිරීම නිසයි.
4,openslr,එවන් ශ්‍රේෂ්ඨ ජාතියක් බිහි කිරීමට


## Observe raw transcripts

Before touching anything: sample raw text, check length distribution, and run a
full alphabet inspection (same idea as the source pipeline's step 14, but run
*first* here as a baseline instead of only at the end) to see what's actually in
the data before deciding what needs cleaning.

In [2]:
N_SAMPLE = 5

print("=== Random sample of raw transcripts ===")
for t in df_trim["text"].fillna("").sample(min(N_SAMPLE, len(df_trim)), random_state=42):
    print(f"  {t!r}")

print("\n=== Transcript length (characters) ===")
print(df_trim["text"].fillna("").str.len().describe())

=== Random sample of raw transcripts ===
  'ඔයාට මේ වැඩේ වැඩි කාලයක් කරගෙන යන්න බැරිවෙයි.'
  'රට බෙදීමේ යෝජනාවයි.'
  'ඊට ඉස්සෙල්ලා අපි හිටියෙ දොළුකන්ද රක්ෂිතයේ'
  'ඔහුගෙම සහොදරයන් විම'
  'මොකද ඇය විවාහක නිසා.'

=== Transcript length (characters) ===
count    150191.000000
mean         26.220832
std          10.428499
min           2.000000
25%          19.000000
50%          25.000000
75%          32.000000
max         132.000000
Name: text, dtype: float64


In [3]:
SINHALA_START, SINHALA_END = 0x0D80, 0x0DFF
ZW_CHARS = {0x200C, 0x200D}  # ZWNJ, ZWJ -- required for Sinhala conjuncts, not disposable


def inspect_alphabet(texts, top_n=40):
    """Reusable version of the source pipeline's step 14: count every unique character
    across the given texts, with its Unicode codepoint/category/name, plus rollups by
    category so contamination (Cn unassigned, Cc control, foreign scripts) is visible
    at a glance without reading the full per-character table.
    """
    counter = Counter()
    for t in texts:
        counter.update(t)

    rows = []
    for ch, n in counter.items():
        cp = ord(ch)
        rows.append({
            "char": ch,
            "codepoint": f"U+{cp:04X}",
            "category": unicodedata.category(ch),
            "name": unicodedata.name(ch, "<unassigned>"),
            "count": n,
            "is_sinhala_block": SINHALA_START <= cp <= SINHALA_END,
        })
    report = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)

    print(f"Unique characters: {len(report)}")
    print("\nBy category:")
    print(report.groupby("category")["count"].agg(["size", "sum"]).rename(columns={"size": "unique_chars", "sum": "total_occurrences"}))

    n_unassigned = (report["category"] == "Cn").sum()
    n_control = ((report["category"] == "Cc") & (report["char"] != "\n")).sum()
    n_space_like = (report["category"] == "Zs").sum()
    print(f"\nUnassigned (Cn) codepoints: {n_unassigned}  <- encoding corruption if > 0")
    print(f"Stray control (Cc, excl. \\n) codepoints: {n_control}  <- corruption if > 0")
    print(f"Distinct space-like (Zs) characters: {n_space_like}  <- should be 1 after whitespace cleanup")

    return report


print("=== Baseline alphabet report (raw text, before any cleaning) ===")
baseline_report = inspect_alphabet(df_trim["text"].fillna(""))
baseline_report.head(40)

=== Baseline alphabet report (raw text, before any cleaning) ===
Unique characters: 108

By category:
          unique_chars  total_occurrences
category                                 
Cc                   1                  4
Cf                   3              27728
Lo                  56            2186327
Mc                  15             461676
Mn                   5             704837
Nd                  10                898
Pd                   1                 15
Pf                   3                130
Pi                   2                116
Po                  11              45504
Zs                   1             510898

Unassigned (Cn) codepoints: 0  <- encoding corruption if > 0
Stray control (Cc, excl. \n) codepoints: 1  <- corruption if > 0
Distinct space-like (Zs) characters: 1  <- should be 1 after whitespace cleanup


,char,codepoint,category,name,count,is_sinhala_block
0,,U+0020,Zs,SPACE,510898,False
1,්,U+0DCA,Mn,SINHALA SIGN AL-LAKUNA,292825,True
2,න,U+0DB1,Lo,SINHALA LETTER DANTAJA NAYANNA,269395,True
3,ි,U+0DD2,Mn,SINHALA VOWEL SIGN KETTI IS-PILLA,240095,True
4,ව,U+0DC0,Lo,SINHALA LETTER VAYANNA,190580,True
5,ක,U+0D9A,Lo,SINHALA LETTER ALPAPRAANA KAYANNA,176114,True
6,ය,U+0DBA,Lo,SINHALA LETTER YAYANNA,168907,True
7,ම,U+0DB8,Lo,SINHALA LETTER MAYANNA,161196,True
8,ත,U+0DAD,Lo,SINHALA LETTER ALPAPRAANA TAYANNA,155048,True
9,ා,U+0DCF,Mc,SINHALA VOWEL SIGN AELA-PILLA,154128,True


## Step 1 — Unicode normalization (NFC)

Same rationale as the source pipeline's step 00: different scrapers/input methods can
represent the same visible Sinhala character sequence with different underlying
codepoint sequences (e.g. a precomposed vs. decomposed combining-mark sequence). NFC
normalization makes every occurrence canonical, so every later regex/character-class
check in this notebook can assume a single consistent representation. Runs first,
before any other step, for the same reason it does in the source pipeline: it doesn't
touch plain ASCII and has no ordering dependency on anything else.

Observe how many rows actually change under NFC before applying it.

In [4]:
raw_text = df_trim["text"].fillna("")
nfc_preview = raw_text.apply(lambda t: unicodedata.normalize("NFC", t))

changed_mask = nfc_preview != raw_text
print(f"Rows that change under NFC normalization: {changed_mask.sum()} out of {len(raw_text)}")

for idx in df_trim[changed_mask].index[:5]:
    print(f"\n[{idx}] before: {raw_text.loc[idx]!r}")
    print(f"[{idx}] after:  {nfc_preview.loc[idx]!r}")

Rows that change under NFC normalization: 0 out of 150191


In [5]:
df_trim["text_v1_nfc"] = nfc_preview
print("Applied -- df_trim['text_v1_nfc'] added. Original 'text' column left untouched.")

Applied -- df_trim['text_v1_nfc'] added. Original 'text' column left untouched.


## Step 2 — Whitespace cleanup

Collapses repeated spaces/tabs, converts non-breaking spaces (`U+00A0`) and other
`Zs`-category space variants to a regular ASCII space, and trims leading/trailing
whitespace. Deliberately does **not** use `str.isspace()` to decide what counts as
whitespace — per the source pipeline's design notes, `isspace()` returns `True` for
some ASCII control characters (e.g. `U+001F`) that are corruption artifacts, not
real whitespace. Only actual Unicode space separators (`Zs` category) plus tab and
newline are treated as collapsible whitespace here.

Observe how many rows have non-standard whitespace before collapsing it.

In [6]:
def is_collapsible_space(ch):
    return unicodedata.category(ch) == "Zs" or ch in ("\t", "\n", "\r")


def has_nonstandard_whitespace(t):
    return any(is_collapsible_space(ch) and ch != " " for ch in t) or t != t.strip() or "  " in t


nonstd_ws_mask = df_trim["text_v1_nfc"].apply(has_nonstandard_whitespace)
print(f"Rows with non-standard whitespace (repeated spaces, non-breaking space, "
      f"leading/trailing whitespace): {nonstd_ws_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[nonstd_ws_mask].index[:5]:
    print(f"[{idx}] {df_trim.loc[idx, 'text_v1_nfc']!r}")

Rows with non-standard whitespace (repeated spaces, non-breaking space, leading/trailing whitespace): 0 out of 150191


In [7]:
def clean_whitespace(t):
    t = "".join(" " if (is_collapsible_space(ch) and ch != "\n") else ch for ch in t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n+", "\n", t)
    return t.strip()


df_trim["text_v2_ws"] = df_trim["text_v1_nfc"].apply(clean_whitespace)

print("Applied -- df_trim['text_v2_ws'] added.")
for idx in df_trim[nonstd_ws_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v1_nfc']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v2_ws']!r}")

Applied -- df_trim['text_v2_ws'] added.


## Step 3 — Punctuation cleanup

Removes empty bracket leftovers (`()`, `[ ]`) and collapses runs of repeated or mixed
punctuation (`...`, `?!`, `--`) down to a single mark — same targeted cleanup as the
source pipeline's step 03/08. Explicitly excludes Sinhala combining marks (`Mn`/`Mc`
categories) and the zero-width joiner/non-joiner from being touched by the
punctuation regex, per the doc's warning that a naive "letters are protected,
everything else is disposable punctuation" approach silently mangles every word
with a vowel sign.

In [8]:
EMPTY_BRACKETS_RE = re.compile(r"\(\s*\)|\[\s*\]|\{\s*\}")
REPEATED_PUNCT_RE = re.compile(r"([.,!?;:\-]){2,}")
# Typographic quotes -> ASCII equivalents, applied before anything else here.
# Deleting these (like other disallowed punctuation) would be safe for quote-mark
# usage (word boundaries already have spaces) but wrong for apostrophes sitting
# mid-word (e.g. a curly-quote possessive/contraction) -- space-replacing those
# splits the word in two. Normalizing to ASCII keeps the character instead.
SMART_QUOTES = {"\u2018": "'", "\u2019": "'", "\u201C": '"', "\u201D": '"'}


def clean_punctuation(t):
    for smart, ascii_eq in SMART_QUOTES.items():
        t = t.replace(smart, ascii_eq)
    t = EMPTY_BRACKETS_RE.sub("", t)
    t = REPEATED_PUNCT_RE.sub(r"\1", t)
    return t


punct_preview = df_trim["text_v2_ws"].apply(clean_punctuation)
punct_changed_mask = punct_preview != df_trim["text_v2_ws"]
print(f"Rows changed by punctuation cleanup: {punct_changed_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[punct_changed_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v2_ws']!r}")
    print(f"[{idx}] after:  {punct_preview.loc[idx]!r}")

Rows changed by punctuation cleanup: 348 out of 150191

[300] before: 'ඒ අය වෙනුවෙන්..'
[300] after:  'ඒ අය වෙනුවෙන්.'

[400] before: 'ඉබේ වෙච්චි දෙයක්..'
[400] after:  'ඉබේ වෙච්චි දෙයක්.'

[446] before: 'උපසිරැසි යෙදුවා..'
[446] after:  'උපසිරැසි යෙදුවා.'

[882] before: 'එළ එළ ජය වේවා!.'
[882] after:  'එළ එළ ජය වේවා.'

[890] before: 'ලිපිවල පෙන්නල දුන්න..'
[890] after:  'ලිපිවල පෙන්නල දුන්න.'


In [9]:
df_trim["text_v3_punct"] = punct_preview
print("Applied -- df_trim['text_v3_punct'] added.")

Applied -- df_trim['text_v3_punct'] added.


## Step 4 — Emoji detection and removal

Speech transcripts are far less likely to contain emoji than web text, but crowd-sourced
or scraped-caption sources can still pick some up. Removes emoji character sequences,
including multi-codepoint emoji joined with ZWJ (e.g. family/flag emoji) — same logic
as the source pipeline's step 06: a ZWJ is only stripped when it sits directly between
two emoji codepoints, so it's never confused with the ZWJ used inside Sinhala
conjuncts (which sits between two Sinhala consonants, not emoji).

In [10]:
# Common emoji Unicode blocks (pictographs, symbols, transport, flags, supplemental
# symbols, dingbats). Deliberately range-based rather than a package dependency, since
# only detection + stripping is needed here, not emoji-aware text shortening/aliasing.
EMOJI_RANGES = [
    (0x1F300, 0x1FAFF),  # misc symbols & pictographs, emoticons, transport, supplemental symbols
    (0x2600, 0x27BF),    # misc symbols, dingbats
    (0x1F1E6, 0x1F1FF),  # regional indicators (flag letters)
    (0x2190, 0x21FF),    # arrows (occasionally used decoratively, low risk here)
    (0xFE0F, 0xFE0F),    # variation selector-16 (emoji presentation)
]


def is_emoji(ch):
    cp = ord(ch)
    return any(lo <= cp <= hi for lo, hi in EMOJI_RANGES)


def has_emoji(t):
    return any(is_emoji(ch) for ch in t)


emoji_mask = df_trim["text_v3_punct"].apply(has_emoji)
print(f"Rows containing emoji: {emoji_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[emoji_mask].index[:5]:
    print(f"[{idx}] {df_trim.loc[idx, 'text_v3_punct']!r}")

Rows containing emoji: 0 out of 150191


In [11]:
def remove_emoji(t):
    out = []
    for i, ch in enumerate(t):
        if is_emoji(ch):
            continue
        # Drop a ZWJ only when it sits directly between two emoji codepoints -- leaves
        # Sinhala-conjunct ZWJ (between two Sinhala consonants) completely untouched.
        if ord(ch) == 0x200D and i > 0 and i + 1 < len(t) and is_emoji(t[i - 1]) and is_emoji(t[i + 1]):
            continue
        out.append(ch)
    return clean_whitespace("".join(out))


df_trim["text_v4_emoji"] = df_trim["text_v3_punct"].apply(remove_emoji)

print("Applied -- df_trim['text_v4_emoji'] added.")
for idx in df_trim[emoji_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v3_punct']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v4_emoji']!r}")

Applied -- df_trim['text_v4_emoji'] added.


## Step 5 — Final character whitelist (safety net)

Same rationale as the source pipeline's step 13: the targeted steps above only
guarantee that the specific patterns they were built for are gone, not what
character set survives overall. This final pass is a categorical safety net against
anything unanticipated — leftover Tamil/Arabic/etc. script fragments,
private-use-area and unassigned codepoints (encoding corruption), stray symbols.

**Whitelist for this project (Sinhala + Latin, not Sinhala-only):**
- Sinhala Unicode block, assigned codepoints only (rejects `Cn` unassigned gaps like
  `U+0DB2`, per the doc's encoding-corruption warning)
- ZWJ / ZWNJ (`U+200D` / `U+200C`) — required for Sinhala conjuncts
- ASCII Latin letters (`A-Za-z`) and digits — kept, unlike the source pipeline,
  because code-mixed English is intentionally retained in this dataset
- A small fixed punctuation set: `. , ! ? ; : ' " ( ) - /`
- Normalized whitespace (single space, single newline)

Same visibility-based removal rule as the source doc: invisible/control characters
(`Cf`, `Cc`, `Cn`) are deleted outright (no replacement), since inserting a visible
space where an invisible character used to be would introduce a word break that
never existed. Visible disallowed characters (foreign scripts, stray symbols) are
replaced with a single space, so two words separated only by the removed character
don't fuse together.

In [12]:
ALLOWED_PUNCTUATION = set(".,!?;:'\"()-/")


def is_allowed_char(ch):
    cp = ord(ch)
    if SINHALA_START <= cp <= SINHALA_END:
        return unicodedata.category(ch) != "Cn"  # reject unassigned gaps in the block
    if cp in ZW_CHARS:
        return True
    if ch.isascii() and (ch.isalpha() or ch.isdigit()):
        return True
    if ch in ALLOWED_PUNCTUATION:
        return True
    if ch in (" ", "\n"):
        return True
    return False


def whitelist_dry_run(t):
    """Returns (would_change, disallowed_chars_found) without modifying anything."""
    disallowed = {ch for ch in t if not is_allowed_char(ch)}
    return bool(disallowed), disallowed


dry_run = df_trim["text_v4_emoji"].apply(whitelist_dry_run)
would_change_mask = dry_run.apply(lambda r: r[0])
all_disallowed = Counter()
for _, disallowed in dry_run:
    all_disallowed.update(disallowed)

print(f"Rows the whitelist would change: {would_change_mask.sum()} out of {len(df_trim)}")
print(f"\nDisallowed characters found (would be stripped), by frequency:")
for ch, n in all_disallowed.most_common(30):
    cp = ord(ch)
    print(f"  {ch!r}  U+{cp:04X}  {unicodedata.category(ch)}  {unicodedata.name(ch, '<unassigned>')}  x{n}")

for idx in df_trim[would_change_mask].index[:5]:
    print(f"\n[{idx}] {df_trim.loc[idx, 'text_v4_emoji']!r}")

Rows the whitelist would change: 33 out of 150191

Disallowed characters found (would be stripped), by frequency:
  '–'  U+2013  Pd  EN DASH  x15
  '\u200b'  U+200B  Cf  ZERO WIDTH SPACE  x6
  '\x94'  U+0094  Cc  <unassigned>  x4
  'ª'  U+00AA  Lo  FEMININE ORDINAL INDICATOR  x3
  '%'  U+0025  Po  PERCENT SIGN  x3
  '»'  U+00BB  Pf  RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK  x2

[2018] 'මෙහි මුද්\u200dරාව අභය හෝ වරද මුද්\u200dරාව යුතුªයැයි සැලකේ'

[10459] 'ඕං ඉතිං පේනව\u200bනේ'

[19535] 'සාපේක්ෂ– වෙනත් දෙයක් මත රඳා පැවතීම'

[26905] 'සාපේක්ෂ– වෙනත් දෙයක් මත රඳා පැවතීම'

[28726] 'දෙමළ ජාතික සන්ධාන – මුස්ලිම් කොංග්\u200dරස්'


In [13]:
INVISIBLE_CATEGORIES = {"Cf", "Cc", "Cn"}


def apply_whitelist(t):
    out = []
    for ch in t:
        if is_allowed_char(ch):
            out.append(ch)
        elif unicodedata.category(ch) in INVISIBLE_CATEGORIES:
            pass  # delete outright -- no replacement, avoids inserting a false word break
        else:
            out.append(" ")  # visible disallowed char -- replace with space, avoids word fusion
    return clean_whitespace("".join(out))


df_trim["text_v5_whitelist"] = df_trim["text_v4_emoji"].apply(apply_whitelist)

print("Applied -- df_trim['text_v5_whitelist'] added.")
for idx in df_trim[would_change_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v4_emoji']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v5_whitelist']!r}")

Applied -- df_trim['text_v5_whitelist'] added.

[2018] before: 'මෙහි මුද්\u200dරාව අභය හෝ වරද මුද්\u200dරාව යුතුªයැයි සැලකේ'
[2018] after:  'මෙහි මුද්\u200dරාව අභය හෝ වරද මුද්\u200dරාව යුතු යැයි සැලකේ'

[10459] before: 'ඕං ඉතිං පේනව\u200bනේ'
[10459] after:  'ඕං ඉතිං පේනවනේ'

[19535] before: 'සාපේක්ෂ– වෙනත් දෙයක් මත රඳා පැවතීම'
[19535] after:  'සාපේක්ෂ වෙනත් දෙයක් මත රඳා පැවතීම'

[26905] before: 'සාපේක්ෂ– වෙනත් දෙයක් මත රඳා පැවතීම'
[26905] after:  'සාපේක්ෂ වෙනත් දෙයක් මත රඳා පැවතීම'

[28726] before: 'දෙමළ ජාතික සන්ධාන – මුස්ලිම් කොංග්\u200dරස්'
[28726] after:  'දෙමළ ජාතික සන්ධාන මුස්ලිම් කොංග්\u200dරස්'


## Verify the output (final alphabet report)

Same check as the source pipeline's step 14, run again here on the cleaned text as
a final QA gate. Compare against the baseline report from the top of this notebook:
unique character count should have dropped sharply, `Cn`/stray `Cc` entries should
be gone, and exactly one space-like (`Zs`) character should remain.

In [14]:
print(f"=== Final alphabet report (cleaned text) ===")
final_report = inspect_alphabet(df_trim["text_v5_whitelist"])

print(f"\nUnique chars: {len(baseline_report)} (baseline) -> {len(final_report)} (cleaned)")

emptied_mask = (df_trim["text_v5_whitelist"].str.strip() == "") & (df_trim["text"].fillna("").str.strip() != "")
print(f"\nRows that became empty after cleaning (whole utterance was junk): {emptied_mask.sum()}")
for idx in df_trim[emptied_mask].index[:10]:
    print(f"  [{idx}] original: {df_trim.loc[idx, 'text']!r}")

final_report.head(40)

=== Final alphabet report (cleaned text) ===
Unique characters: 98

By category:
          unique_chars  total_occurrences
category                                 
Cf                   2              27722
Lo                  55            2186324
Mc                  15             461676
Mn                   5             704837
Nd                  10                898
Po                  10              45515
Zs                   1             510891

Unassigned (Cn) codepoints: 0  <- encoding corruption if > 0
Stray control (Cc, excl. \n) codepoints: 0  <- corruption if > 0
Distinct space-like (Zs) characters: 1  <- should be 1 after whitespace cleanup

Unique chars: 108 (baseline) -> 98 (cleaned)

Rows that became empty after cleaning (whole utterance was junk): 0


,char,codepoint,category,name,count,is_sinhala_block
0,,U+0020,Zs,SPACE,510891,False
1,්,U+0DCA,Mn,SINHALA SIGN AL-LAKUNA,292825,True
2,න,U+0DB1,Lo,SINHALA LETTER DANTAJA NAYANNA,269395,True
3,ි,U+0DD2,Mn,SINHALA VOWEL SIGN KETTI IS-PILLA,240095,True
4,ව,U+0DC0,Lo,SINHALA LETTER VAYANNA,190580,True
5,ක,U+0D9A,Lo,SINHALA LETTER ALPAPRAANA KAYANNA,176114,True
6,ය,U+0DBA,Lo,SINHALA LETTER YAYANNA,168907,True
7,ම,U+0DB8,Lo,SINHALA LETTER MAYANNA,161196,True
8,ත,U+0DAD,Lo,SINHALA LETTER ALPAPRAANA TAYANNA,155048,True
9,ා,U+0DCF,Mc,SINHALA VOWEL SIGN AELA-PILLA,154128,True


## Step 6 — Pure English transcripts (flag for removal, expected ~0)

This is a Sinhala ASR corpus; per the notebook intro above, `audio_openslr.ipynb`
already dropped pure-English clips (no Sinhala script at all) upstream, at the
audio-cleaning stage. This re-runs the identical check as a categorical safety
net -- same call already made for Linga/YouTube/BizBrains in `transcript.ipynb`,
applied here for consistency across all four sources -- not because it's expected
to find anything. Flagged on the fully-cleaned `text_v5_whitelist` column; actually
dropped together with the emptied rows in the commit step below.

In [15]:
ENGLISH_LETTER_RE = re.compile(r"[A-Za-z]")


def is_pure_english(t):
    has_sinhala = any(SINHALA_START <= ord(ch) <= SINHALA_END for ch in t)
    has_english = bool(ENGLISH_LETTER_RE.search(t))
    return (not has_sinhala) and has_english


pure_english_mask = df_trim["text_v5_whitelist"].apply(is_pure_english)
print(f"Pure English rows (no Sinhala script, has Latin letters): "
      f"{pure_english_mask.sum()} out of {len(df_trim)}")
df_trim.loc[pure_english_mask, ["source_dataset", "text_v5_whitelist"]].head(10)

Pure English rows (no Sinhala script, has Latin letters): 0 out of 150191


,source_dataset,text_v5_whitelist


## Commit cleaned text + drop emptied and pure-English rows

**Status: applied.** Promotes `text_v5_whitelist` -> `text`, drops rows that became
empty after cleaning and rows flagged as pure English above, recomputes `text_len`,
and drops the intermediate `text_v1`...`text_v5_whitelist` columns -- same as the
source notebook's commit step.

Writes the result to a **new companion TSV**, `utt_spk_text_cleaned.tsv`, alongside
the existing `utt_spk_text.tsv` in `openslr_52_clean/` -- it does not overwrite the
original TSV and does not delete any `.flac` file, even for rows dropped here (unlike
`audio_openslr.ipynb`'s own pure-English removal, which does delete the `.flac`).
If anything is actually dropped, the printed `file_id`s are for you to reconcile with
the audio tree manually, since deleting audio files isn't something this notebook
does silently.

In [16]:
drop_mask = emptied_mask | pure_english_mask
dropped_file_ids = df_trim.loc[drop_mask, "file_id"].tolist()

df_trim = df_trim[~drop_mask].reset_index(drop=True)
df_trim["text"] = df_trim["text_v5_whitelist"]
df_trim["text_len"] = df_trim["text"].str.len()
df_trim = df_trim.drop(columns=["text_v1_nfc", "text_v2_ws", "text_v3_punct", "text_v4_emoji", "text_v5_whitelist"])
print(f"Dropped {emptied_mask.sum()} emptied rows and {pure_english_mask.sum()} pure-English rows "
      f"-- new shape: {df_trim.shape}")
if dropped_file_ids:
    print(f"Dropped file_ids (still present as .flac files -- not deleted): {dropped_file_ids[:20]}"
          f"{' ...' if len(dropped_file_ids) > 20 else ''}")

out_path = os.path.join(CLEAN_DIR, "utt_spk_text_cleaned.tsv")
df_trim[["file_id", "speaker_id", "source_dataset", "text", "text_len"]].to_csv(out_path, sep="\t", index=False)
print(f"Saved cleaned transcripts to {out_path}")

Dropped 0 emptied rows and 0 pure-English rows -- new shape: (150191, 5)
Saved cleaned transcripts to ../../../../data/raw/openslr_52/processed/openslr_52_clean/utt_spk_text_cleaned.tsv


## Verify the saved cleaned TSV

Reads `utt_spk_text_cleaned.tsv` back from disk (not the in-memory `df_trim`) as a
final sanity check that what got written matches what was intended.

In [17]:
cleaned_path = os.path.join(CLEAN_DIR, "utt_spk_text_cleaned.tsv")
df_clean_openslr = pd.read_csv(cleaned_path, sep="\t")

print(f"Path: {cleaned_path}")
print(f"Shape: {df_clean_openslr.shape[0]} rows x {df_clean_openslr.shape[1]} columns")
print(f"\nColumns and dtypes:")
print(df_clean_openslr.dtypes)

print(f"\ntext_len summary:")
print(df_clean_openslr["text_len"].describe())

print(f"\nNull counts per column:")
print(df_clean_openslr.isnull().sum())

df_clean_openslr.head(5)

Path: ../../../../data/raw/openslr_52/processed/openslr_52_clean/utt_spk_text_cleaned.tsv
Shape: 150191 rows x 5 columns

Columns and dtypes:
file_id             str
speaker_id          str
source_dataset      str
text                str
text_len          int64
dtype: object

text_len summary:
count    150191.000000
mean         26.219034
std          10.428842
min           2.000000
25%          19.000000
50%          25.000000
75%          32.000000
max         132.000000
Name: text_len, dtype: float64

Null counts per column:
file_id           0
speaker_id        0
source_dataset    0
text              0
text_len          0
dtype: int64


,file_id,speaker_id,source_dataset,text,text_len
0,0000f47c22,7ab05,openslr,මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී,30
1,000101700f,44e28,openslr,උන්වහන්සේ කපාපු,15
2,000107b539,b1a64,openslr,එය එතනින් අවසන් නොවී,20
3,00016825d3,2fff2,openslr,සිතින් අයහපතෙහි හැසිරීම නිසයි.,30
4,000171b8fd,d6ccd,openslr,එවන් ශ්‍රේෂ්ඨ ජාතියක් බිහි කිරීමට,33


## Export final dataset

Same three columns and same names as `transcript.ipynb`'s `final_dataset.parquet`
(`audio`, `source_dataset`, `text`) so the two are consistent and easy to concatenate
later. The only difference is what `audio` holds: the other notebook embeds raw WAV
*bytes* (its source was already an embedded-bytes parquet); here `audio` is a
repo-root-relative **path** to the `.flac` file, since OpenSLR-52 was deliberately
kept as loose files on disk (see the intro note on why bytes weren't embedded).
Written as `final_dataset_openslr.tsv` into the same shared
`model-development/data/final_dataset/` folder the other notebook writes into.

In [ ]:

os.makedirs(FINAL_DIR, exist_ok=True)

AUDIO_ROOT_REL = "data/raw/openslr_52/processed/openslr_52_clean/data"  # relative to repo root


def flac_repo_relative_path(file_id):
    return f"{AUDIO_ROOT_REL}/{file_id[:2]}/{file_id}.flac"


final_dataset = pd.DataFrame({
    "audio": df_clean_openslr["file_id"].apply(flac_repo_relative_path),
    "source_dataset": df_clean_openslr["source_dataset"],
    "text": df_clean_openslr["text"],
})

final_path = os.path.join(FINAL_DIR, "final_dataset_openslr.tsv")
final_dataset.to_csv(final_path, sep="\t", index=False, encoding="utf-8")

print(f"Saved final dataset ({len(final_dataset)} rows) to {final_path}")
print(f"\nRows per source_dataset:")
print(final_dataset["source_dataset"].value_counts())
final_dataset.head(5)

## Export final dataset (parquet, embedded WAV bytes)

Same file the other three sources produce -- `audio` (bytes), `source_dataset`, `text`
-- written into the same `model-development/data/final_dataset/` folder as
`final_dataset_openslr.parquet`. `audio` holds re-encoded WAV bytes (not raw `.flac`
bytes), matching the WAV format already embedded in `final_dataset.parquet` for
Linga/YouTube/BizBrains -- same conversion `scripts/convert_openslr.py` already does
for this exact corpus, so schemas and audio encoding line up.

**On the OOM concern raised earlier:** this uses the same fix `convert_openslr.py`
and `audio_openslr.ipynb` already use for this dataset -- write in bounded batches
via `pyarrow.parquet.ParquetWriter` (one row-group per batch) instead of building a
150k-row list of WAV blobs in memory before writing anything. Peak memory is capped
at one batch (`BATCH_SIZE` rows), not the whole dataset.

In [20]:
import io

import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf

FINAL_DIR = "../../data/final_dataset"

FINAL_PARQUET_PATH = os.path.join(FINAL_DIR, "final_dataset_openslr.parquet")
BATCH_SIZE = 2000  # rows per row-group -- bounds peak memory instead of holding all ~150k WAV blobs at once

parquet_schema = pa.schema([
    pa.field("audio", pa.binary()),
    pa.field("source_dataset", pa.string()),
    pa.field("text", pa.string()),
])


def flac_to_wav_bytes(file_id):
    flac_path = os.path.join(CLEAN_DIR, "data", file_id[:2], f"{file_id}.flac")
    data, sr = sf.read(flac_path, dtype="int16", always_2d=False)
    buf = io.BytesIO()
    sf.write(buf, data, sr, format="WAV", subtype="PCM_16")
    return buf.getvalue()


n_rows = len(df_clean_openslr)
n_written = 0

writer = pq.ParquetWriter(FINAL_PARQUET_PATH, parquet_schema)
try:
    for start in range(0, n_rows, BATCH_SIZE):
        batch = df_clean_openslr.iloc[start:start + BATCH_SIZE]
        audio_bytes = [flac_to_wav_bytes(fid) for fid in batch["file_id"]]
        table = pa.table(
            {
                "audio": pa.array(audio_bytes, type=pa.binary()),
                "source_dataset": pa.array(batch["source_dataset"], type=pa.string()),
                "text": pa.array(batch["text"], type=pa.string()),
            },
            schema=parquet_schema,
        )
        writer.write_table(table)
        n_written += len(batch)
        print(f"  wrote rows {start}-{start + len(batch)} ({n_written}/{n_rows})")
finally:
    writer.close()

print(f"\nSaved final parquet dataset ({n_written} rows) to {FINAL_PARQUET_PATH}")
print(f"File size: {os.path.getsize(FINAL_PARQUET_PATH) / 1e6:.1f} MB")

  wrote rows 0-2000 (2000/150191)
  wrote rows 2000-4000 (4000/150191)
  wrote rows 4000-6000 (6000/150191)
  wrote rows 6000-8000 (8000/150191)
  wrote rows 8000-10000 (10000/150191)
  wrote rows 10000-12000 (12000/150191)
  wrote rows 12000-14000 (14000/150191)
  wrote rows 14000-16000 (16000/150191)
  wrote rows 16000-18000 (18000/150191)
  wrote rows 18000-20000 (20000/150191)
  wrote rows 20000-22000 (22000/150191)
  wrote rows 22000-24000 (24000/150191)
  wrote rows 24000-26000 (26000/150191)
  wrote rows 26000-28000 (28000/150191)
  wrote rows 28000-30000 (30000/150191)
  wrote rows 30000-32000 (32000/150191)
  wrote rows 32000-34000 (34000/150191)
  wrote rows 34000-36000 (36000/150191)
  wrote rows 36000-38000 (38000/150191)
  wrote rows 38000-40000 (40000/150191)
  wrote rows 40000-42000 (42000/150191)
  wrote rows 42000-44000 (44000/150191)
  wrote rows 44000-46000 (46000/150191)
  wrote rows 46000-48000 (48000/150191)
  wrote rows 48000-50000 (50000/150191)
  wrote rows 500